# VCC 2026 data explorer

In [ ]:
import sys, json, math, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, h5py, anndata as ad, scipy.sparse as sp
import matplotlib.pyplot as plt
from scipy.special import expit
from IPython.display import display, Markdown
try:
    from anndata.io import read_elem
except ImportError:
    from anndata.experimental import read_elem
warnings.filterwarnings('ignore', category=RuntimeWarning)

V = Path.home() / 'vcc'
DATA, LIB = V / 'data', V / 'lib'
WORK = LIB / 'review_v4'                      # the pipeline's working folder (read-only here)
CACHE = LIB / 'explore'; CACHE.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(V / 'code'))            # reuse the pipeline's own helpers
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.precision', 4)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25})

GENES = pd.read_csv(DATA / 'gene_names.csv')['gene_name'].astype(str).to_numpy()
GENE_POS = {g: i for i, g in enumerate(GENES)}
PANEL = pd.read_csv(DATA / 'pert_counts.csv')['target_gene'].astype(str).to_numpy()
CONTROL = 'non-targeting'
PC = 0.5                                       # CPM pseudocount for exploratory log fold changes
MIN_CELLS = 20                                 # same minimum the pipeline uses; smaller rows are mostly noise

# key -> (file, storage kind, biological background)
SOURCES = {
    'HCT116':     (LIB / 'orion/HCT116_pseudobulk.h5ad',      'sums',    'HCT116'),
    'HEK293T':    (LIB / 'orion/HEK293T_pseudobulk.h5ad',     'sums',    'HEK293T'),
    'CD4_Stim48': (LIB / 'tcell/TCELL_Stim48hr_effects.h5ad', 'effects', 'CD4_T'),
    'CD4_Rest':   (LIB / 'tcell/TCELL_Rest_effects.h5ad',     'effects', 'CD4_T'),
    'H1':         (LIB / 'h1/H1_full_2025.h5ad',              'cells',   'H1'),
}
LINES = {k: (DATA / f'context_{k}.h5ad', 'cells', f'line {k}') for k in 'ABC'}
ALL = {**SOURCES, **LINES}
missing = [str(p) for p, _, _ in ALL.values() if not p.exists()]
print(f'{len(GENES):,} genes | {len(PANEL)} challenge targets | cache: {CACHE}')
print('missing files:', missing or 'none')

## 1. Header and footer of every file

Each `.h5ad` holds a cell (or row) table `obs`, a gene table `var`, a matrix `X`, and optionally extra matrices in `layers`. Files are opened in **backed** mode, so only the rows shown are read from disk.

What each file stores:

| Context | Rows are | `X` / layers |
|---|---|---|
| HCT116, HEK293T | one row per knockdown, plus one aggregated control row | layers `sum_counts`, `sum_cpm`, `sum_cpm2`, `n_expr`: per-gene sums over that row's cells |
| CD4_T Stim48hr / Rest | one row per knockdown | `X` = log2 fold change vs control; layer `se`; control CPM in `var['ctrl_cpm']` |
| H1 | one row per **cell** | `X` = raw UMI counts |
| A, B, C | one row per **control cell** | `X` = raw UMI counts |

In [ ]:
def _block(matrix, rows, cols, gene_names):
    m = matrix[rows[0]:rows[1]]
    m = m.toarray() if sp.issparse(m) else np.asarray(m)
    return pd.DataFrame(m[:, cols[0]:cols[1]], columns=gene_names[cols[0]:cols[1]])

def peek(name, n=5, k=8):
    path, kind, background = ALL[name]
    display(Markdown(f'### {name}  ·  `{path.name}`  ·  {path.stat().st_size/1e9:.2f} GB  ·  background: {background}'))
    a = ad.read_h5ad(path, backed='r')
    try:
        genes = a.var_names.astype(str).to_numpy()
        x_kind = type(a.X).__name__
        print(f'shape {a.n_obs:,} rows x {a.n_vars:,} genes | X: {x_kind} | layers: {[k for k in a.layers.keys() if k]} | uns: {list(a.uns.keys())}')
        print(f'genes also in the 2026 list: {np.isin(genes, GENES).sum():,} / {len(GENES):,}')
        display(Markdown('**obs (row table): first and last rows**')); display(pd.concat([a.obs.head(n), a.obs.tail(n)]))
        display(Markdown('**var (gene table): first and last genes**')); display(pd.concat([a.var.head(n), a.var.tail(n)]))
        mats = {'X': a.X} if kind != 'sums' else {'layers/sum_counts': a.layers['sum_counts'], 'layers/n_expr': a.layers['n_expr']}
        if kind == 'filtered': mats['layers/se'] = a.layers['se']      # log2FC and its standard error
        for label, m in mats.items():
            display(Markdown(f'**{label}: first {n} rows x first {k} genes, then last {n} rows x last {k} genes**'))
            head = _block(m, (0, n), (0, k), genes); head.index = a.obs_names[:n]
            tail = _block(m, (a.n_obs - n, a.n_obs), (a.n_vars - k, a.n_vars), genes); tail.index = a.obs_names[-n:]
            display(head); display(tail)
        for key, value in a.uns.items():
            print(f'uns[{key!r}]: {str(value)[:300]}')
    finally:
        a.file.close()

display(Markdown('### gene_names.csv and pert_counts.csv'))
display(pd.concat([pd.read_csv(DATA/'gene_names.csv').head(3), pd.read_csv(DATA/'gene_names.csv').tail(3)]))
display(pd.concat([pd.read_csv(DATA/'pert_counts.csv').head(3), pd.read_csv(DATA/'pert_counts.csv').tail(3)]))

### Filtered datasets (`lib/filters/`)

One file per training dataset, built by `scripts/knockdown_filters.py`. Rows are the knockdowns that passed all three filters (at least 100 cells, target expressed in that dataset's controls, target dropped by at least 50% with adjusted p < 0.05). Columns are the challenge genes the dataset measured. `X` is the log2 fold change vs that dataset's controls. `obs` holds each knockdown's QC numbers and `var` each gene's control expression.

In [ ]:
# Self-contained: does not depend on peek(), so it always shows both the log2FC and the SE tables.
def peek_filtered(path, n=5, k=8):
    a = ad.read_h5ad(path, backed='r')
    try:
        genes = a.var_names.astype(str).to_numpy()
        display(Markdown(f'### {path.stem}  ·  `{path.name}`  ·  {path.stat().st_size/1e9:.2f} GB'))
        print(f'{a.n_obs:,} kept knockdowns x {a.n_vars:,} genes | layers: {list(a.layers.keys())}')
        display(Markdown('**obs: one row per knockdown (first and last)**')); display(pd.concat([a.obs.head(n), a.obs.tail(n)]))
        display(Markdown('**var: one row per gene, control expression (first and last)**')); display(pd.concat([a.var.head(n), a.var.tail(n)]))
        for label, m in (('X = log2 fold change vs controls', a.X), ('layers/se = standard error of that log2FC', a.layers['se'])):
            display(Markdown(f'**{label}: first {n} knockdowns x first {k} genes, then last {n} x last {k}**'))
            head = pd.DataFrame(np.asarray(m[:n, :k]), index=a.obs_names[:n], columns=genes[:k])
            tail = pd.DataFrame(np.asarray(m[a.n_obs - n:, a.n_vars - k:]), index=a.obs_names[-n:], columns=genes[-k:])
            display(head); display(tail)
    finally:
        a.file.close()

FILTERED = sorted((LIB / 'filters').glob('*.h5ad'))
print('filtered files:', [p.name for p in FILTERED] or 'none: run scripts/knockdown_filters.py first')
for path in FILTERED:
    peek_filtered(path)


### Panel tables (`lib/sources/`)

One table per source, built by `scripts/build_panel_sources.py`. Every table has the same shape: 575 panel targets (our 300, then H1's other targets) × the 18,533 genes in the 2026 order. It has two layers, `lfc` and `se`, and `X` is empty. A whole row is NaN when the knockdown failed filters 1–3 or wasn't screened (`obs.status` says which). A whole column is NaN when the gene isn't measured or is below 1 CPM in that source's controls. `coverage.csv` lists, per target, which sources kept it.

In [ ]:
# Self-contained: prints head and tail of obs, var, lfc and se for every panel table, then the coverage table.
def peek_panel(path, n=5, k=8):
    a = ad.read_h5ad(path, backed='r')
    try:
        genes = a.var_names.astype(str).to_numpy()
        kept = int((a.obs['status'] == 'kept').sum()); measurable = int(a.var['ctrl_cpm'].ge(1).sum())
        display(Markdown(f'### {path.parent.name}  ·  `{path.parent.name}/{path.name}`  ·  {path.stat().st_size/1e9:.2f} GB'))
        print(f'{a.n_obs:,} targets x {a.n_vars:,} genes | layers: {list(a.layers.keys())} | '
              f'{kept} targets kept, {measurable:,} measurable genes')
        display(Markdown('**obs: one row per panel target (first and last)**')); display(pd.concat([a.obs.head(n), a.obs.tail(n)]))
        display(Markdown('**var: one row per gene, control expression (first and last)**')); display(pd.concat([a.var.head(n), a.var.tail(n)]))
        for key, label in (('lfc', 'log2 fold change vs controls'), ('se', 'standard error of that log2FC')):
            m = a.layers[key]
            display(Markdown(f'**layers/{key} = {label}: first {n} targets x first {k} genes, then last {n} x last {k}**'))
            display(pd.DataFrame(np.asarray(m[:n, :k]), index=a.obs_names[:n], columns=genes[:k]))
            display(pd.DataFrame(np.asarray(m[a.n_obs - n:, a.n_vars - k:]), index=a.obs_names[-n:], columns=genes[-k:]))
    finally:
        a.file.close()

PANELS = sorted((LIB / 'sources').glob('*/effects_panel.h5ad'))
print('panel files:', [str(p.relative_to(LIB)) for p in PANELS] or 'none: run scripts/build_panel_sources.py first')
for path in PANELS:
    peek_panel(path)

COVERAGE = pd.read_csv(LIB / 'sources/coverage.csv', index_col='target')
display(Markdown('### `coverage.csv`: which sources kept each target (first and last)'))
display(pd.concat([COVERAGE.head(), COVERAGE.tail()]))


## 2. Pseudobulk every context, then look at noise and spread

**Pseudobulk** = one expression profile per group of cells. For every context we build a matrix with one row per group (row 0 is always the control population, then one row per knockdown) and one column per gene in the 18,533-gene challenge list. Genes a context did not measure are `NaN`.

Per row and gene we keep:

* `mean_cpm`: mean over the group's cells of per-cell CPM, $\bar y_{pg} = \frac{1}{n_p}\sum_{c\in p} 10^6\,x_{cg}/L_c$. This is the evaluator's DE convention.
* `pooled_cpm`: ratio of sums, $10^6\sum_{c\in p}x_{cg} / \sum_{c\in p}L_c$. This is what the transfer model uses as a baseline.
* `detection`: fraction of the group's cells with at least one count.

How each kind of file becomes a pseudobulk:

| Kind | Source of each row |
|---|---|
| sums (Orion) | `sum_cpm / n_cells`, `sum_counts / row total`, `n_expr / n_cells`. Exact. |
| cells (H1, A, B, C) | streamed from raw counts, 4,096 cells at a time. Exact. |
| effects (T cells) | **reconstructed**: $(\text{ctrl\_cpm}+0.5)\,2^{\text{LFC}}-0.5$, floored at 0, because no cells were released. No detection. |

Results are saved to `~/vcc/lib/explore/pseudobulk/<context>/` as `.npy` arrays you can memory-map later.

In [ ]:
PB = CACHE / 'pseudobulk'

def _gene_map(names):
    pos = {g: i for i, g in enumerate(np.asarray(names, dtype=str))}
    return np.array([pos.get(g, -1) for g in GENES], dtype=np.int64)

def _aligned(block, src):
    out = np.full((block.shape[0], len(GENES)), np.nan, dtype=np.float32)
    ok = src >= 0; out[:, ok] = block[:, src[ok]]; return out

def _save_meta(out, **meta):
    (out / 'meta.json').write_text(json.dumps(meta, indent=1, default=float))

def build_sums(key, path, chunk=512):
    out = PB / key; out.mkdir(parents=True, exist_ok=True)
    with h5py.File(path, 'r') as f:
        obs, var = read_elem(f['obs']), read_elem(f['var'])
        names = obs.index.astype(str).to_numpy(); n = obs['n_cells'].to_numpy(float)
        src = _gene_map(var.index.astype(str)); ok = src >= 0
        measured = np.zeros(len(GENES), bool)
        measured[ok] = var['measured'].to_numpy(bool)[src[ok]] if 'measured' in var else True
        ctrl = int(np.flatnonzero(names == CONTROL)[0])
        out_row = np.empty(len(names), int); out_row[ctrl] = 0
        out_row[np.arange(len(names)) != ctrl] = np.arange(1, len(names))
        arrays = {k: np.lib.format.open_memmap(out / f'{k}.npy', 'w+', np.float32, (len(names), len(GENES)))
                  for k in ('mean_cpm', 'pooled_cpm', 'detection')}
        umi = np.zeros(len(names)); ngenes = np.zeros(len(names)); ctrl_var = None
        L = {k: f[f'layers/{k}'] for k in ('sum_counts', 'sum_cpm', 'sum_cpm2', 'n_expr')}
        for b in range(0, len(names), chunk):
            sl = slice(b, min(b + chunk, len(names))); nn = n[sl][:, None]
            cnt = L['sum_counts'][sl].astype(np.float64); tot = cnt.sum(1, keepdims=True)
            mean = L['sum_cpm'][sl] / nn; det = L['n_expr'][sl] / nn
            rows = out_row[sl]
            for k, v in (('mean_cpm', mean), ('pooled_cpm', cnt / np.maximum(tot, 1) * 1e6), ('detection', det)):
                al = _aligned(v, src); al[:, ~measured] = np.nan; arrays[k][rows] = al
            umi[rows] = tot[:, 0] / n[sl]; ngenes[rows] = (cnt > 0).sum(1)
            if sl.start <= ctrl < sl.stop:
                i = ctrl - sl.start
                m2 = L['sum_cpm2'][ctrl] / n[ctrl]
                ctrl_var = _aligned((m2 - mean[i] ** 2)[None, :], src)[0]
            print(f'  {key}: {sl.stop:,}/{len(names):,}', end='\r')
        for a_ in arrays.values(): a_.flush()
    targets = np.empty(len(names), dtype=object); targets[out_row] = names
    np.save(out / 'targets.npy', targets.astype(str)); np.save(out / 'n_cells.npy', n[np.argsort(out_row)])
    np.save(out / 'umi_per_cell.npy', umi); np.save(out / 'genes_detected.npy', ngenes)
    np.save(out / 'ctrl_var_cpm.npy', np.maximum(ctrl_var, 0)); np.save(out / 'measured.npy', measured)
    _save_meta(out, kind='sums', source=str(path), exact=True)
    (out / 'done').touch(); print(f'\n  {key}: wrote {out}')

def build_cells(key, path, obs_col='target_gene', block=4096):
    out = PB / key; out.mkdir(parents=True, exist_ok=True)
    a = ad.read_h5ad(path, backed='r')
    try:
        labels = a.obs[obs_col].astype(str).to_numpy() if obs_col in a.obs else np.full(a.n_obs, CONTROL)
        groups = np.array([CONTROL] + sorted(set(labels) - {CONTROL}))
        code = pd.Index(groups).get_indexer(labels); k, G = len(groups), a.n_vars
        n = np.bincount(code, minlength=k).astype(float)
        s_cnt, s_cpm, s_cpm2, s_det = (np.zeros((k, G)) for _ in range(4))
        lib = np.zeros(a.n_obs); ngen = np.zeros(a.n_obs, np.int32)
        for start in range(0, a.n_obs, block):
            m = a.X[start:start + block]
            m = (m.tocsr() if sp.issparse(m) else sp.csr_matrix(np.asarray(m))).astype(np.float64)
            l = np.asarray(m.sum(1)).ravel(); stop = start + len(l)
            lib[start:stop] = l; ngen[start:stop] = np.diff(m.indptr)
            ind = sp.csr_matrix((np.ones(len(l)), (code[start:stop], np.arange(len(l)))), shape=(k, len(l)))
            cpm = sp.diags(1e6 / np.maximum(l, 1)) @ m
            s_cnt += (ind @ m).toarray(); s_cpm += (ind @ cpm).toarray()
            s_cpm2 += (ind @ cpm.multiply(cpm)).toarray(); s_det += (ind @ (m > 0).astype(np.float64)).toarray()
            print(f'  {key}: {stop:,}/{a.n_obs:,} cells', end='\r')
        src = _gene_map(a.var_names.astype(str))
    finally:
        a.file.close()
    mean = s_cpm / n[:, None]
    np.save(out / 'mean_cpm.npy', _aligned(mean, src))
    np.save(out / 'pooled_cpm.npy', _aligned(s_cnt / np.maximum(s_cnt.sum(1, keepdims=True), 1) * 1e6, src))
    np.save(out / 'detection.npy', _aligned(s_det / n[:, None], src))
    np.save(out / 'var_cpm.npy', _aligned(np.maximum(s_cpm2 / n[:, None] - mean ** 2, 0), src))
    np.save(out / 'ctrl_var_cpm.npy', _aligned(np.maximum(s_cpm2[:1] / n[0] - mean[:1] ** 2, 0), src)[0])
    np.save(out / 'targets.npy', groups.astype(str)); np.save(out / 'n_cells.npy', n)
    np.save(out / 'umi_per_cell.npy', np.bincount(code, weights=lib, minlength=k) / n)
    np.save(out / 'genes_detected.npy', (s_cnt > 0).sum(1))
    np.save(out / 'cell_library.npy', lib); np.save(out / 'cell_genes.npy', ngen); np.save(out / 'cell_group.npy', code.astype(np.int32))
    np.save(out / 'measured.npy', src >= 0)
    _save_meta(out, kind='cells', source=str(path), exact=True)
    (out / 'done').touch(); print(f'\n  {key}: wrote {out}')

def build_effects(key, path, chunk=1024):
    out = PB / key; out.mkdir(parents=True, exist_ok=True)
    with h5py.File(path, 'r') as f:
        obs, var = read_elem(f['obs']), read_elem(f['var'])
        names = obs.index.astype(str).to_numpy(); src = _gene_map(var.index.astype(str))
        measured = np.zeros(len(GENES), bool); ok = src >= 0
        measured[ok] = var['measured'].to_numpy(bool)[src[ok]] if 'measured' in var else True
        ctrl = _aligned(var['ctrl_cpm'].to_numpy(float)[None, :], src)[0]; ctrl[~measured] = np.nan
        keep = np.flatnonzero(names != CONTROL)
        mean = np.lib.format.open_memmap(out / 'mean_cpm.npy', 'w+', np.float32, (len(keep) + 1, len(GENES)))
        mean[0] = ctrl; se_sum = np.zeros(len(GENES)); se_n = np.zeros(len(GENES)); se_med = np.zeros(len(keep))
        X, S = f['X'], f['layers/se']
        for b in range(0, len(keep), chunk):
            rows = keep[b:b + chunk]
            lfc = _aligned(X[rows[0]:rows[-1] + 1][rows - rows[0]], src)
            se = _aligned(S[rows[0]:rows[-1] + 1][rows - rows[0]], src)
            mean[1 + b:1 + b + len(rows)] = np.maximum((ctrl + PC) * np.exp2(lfc) - PC, 0)
            good = np.isfinite(se); se_sum += np.where(good, se, 0).sum(0); se_n += good.sum(0)
            se_med[b:b + len(rows)] = np.nanmedian(se, axis=1)
        mean.flush()
    np.save(out / 'targets.npy', np.r_[[CONTROL], names[keep]].astype(str))
    np.save(out / 'n_cells.npy', np.r_[[np.nan], obs['n_cells'].to_numpy(float)[keep]])
    np.save(out / 'n_reps.npy', np.r_[[np.nan], obs['n_reps'].to_numpy(float)[keep]])
    np.save(out / 'mean_se.npy', se_sum / np.maximum(se_n, 1)); np.save(out / 'target_median_se.npy', se_med)
    np.save(out / 'measured.npy', measured)
    _save_meta(out, kind='effects', source=str(path), exact=False,
               note='mean_cpm reconstructed as (ctrl_cpm+0.5)*2^LFC-0.5; no cells, so no detection or per-cell variance')
    (out / 'done').touch(); print(f'  {key}: wrote {out}')

def ensure_pseudobulk(key):
    path, kind, _ = ALL[key]
    if (PB / key / 'done').exists(): return PB / key
    t0 = time.time(); print(f'building {key} ({kind}) ...')
    {'sums': build_sums, 'cells': build_cells, 'effects': build_effects}[kind](key, path)
    print(f'  {key}: {time.time() - t0:.0f} s'); return PB / key

def load_pb(key):
    d = ensure_pseudobulk(key)
    pb = {'key': key, 'kind': ALL[key][1], 'dir': d}
    for f in d.glob('*.npy'):
        pb[f.stem] = np.load(f, mmap_mode='r', allow_pickle=False)
    pb['meta'] = json.loads((d / 'meta.json').read_text())
    return pb

In [ ]:
# Builds anything not cached yet. H1 (29 GB, 414,694 cells) dominates the first run.
for key in ALL:
    ensure_pseudobulk(key)
PBS = {key: load_pb(key) for key in ALL}

### 2a. Size and depth of each context

* **Cells per knockdown** sets how noisy each knockdown's pseudobulk is (standard error shrinks like $1/\sqrt{n}$).
* **UMIs per cell** (library size) sets how many genes are measurable at all: at 20,000 UMIs, 5 CPM is 0.1 counts per cell.
* **Genes above 5 CPM in controls** is exactly the evaluator's DE gene set for that context.

In [ ]:
def context_table():
    rows = []
    for key, pb in PBS.items():
        n = np.asarray(pb['n_cells'])[1:]; ctrl = np.asarray(pb['mean_cpm'][0]); meas = np.asarray(pb['measured'])
        rows.append(dict(context=key, kind=pb['kind'], knockdowns=len(n),
                         control_cells=float(np.asarray(pb['n_cells'])[0]),
                         cells_per_kd_median=np.nanmedian(n) if len(n) else np.nan,
                         cells_per_kd_iqr=f'{np.nanpercentile(n,25):.0f}-{np.nanpercentile(n,75):.0f}' if len(n) and np.isfinite(n).any() else '',
                         ctrl_umi_per_cell=float(np.asarray(pb['umi_per_cell'])[0]) if 'umi_per_cell' in pb else np.nan,
                         genes_measured=int(meas.sum()),
                         genes_ctrl_gt5cpm=int(np.nansum(ctrl > 5)),
                         panel_targets_present=int(np.isin(PANEL, np.asarray(pb['targets'])).sum())))
    return pd.DataFrame(rows).set_index('context')
SUMMARY = context_table(); SUMMARY

### 2b. Noise and spread, context by context

Four panels per context:

1. **Cells per knockdown** (log scale).
2. **Depth**: UMIs per cell, per cell for cell-level contexts and per knockdown average for Orion.
3. **Control mean–variance**: each dot is a gene. The dashed line is pure Poisson counting noise at the median depth, $\operatorname{Var}(y)\approx \bar y\cdot 10^6/L$. Genes above it are **overdispersed**: biological cell-to-cell variability on top of counting noise.
4. **Signal vs noise across knockdowns** (knockdowns with at least 20 cells), binned by control expression. The blue line is the median **across-knockdown spread**, the SD over knockdowns of $\log_2\frac{\bar y_{pg}+0.5}{\bar y_{0g}+0.5}$. The orange line is the median **expected sampling SE** of that same quantity for a typical knockdown. Where blue ≈ orange, differences between knockdowns are mostly noise. Where blue ≫ orange, there is real knockdown-specific signal.

In [ ]:
def kd_rows(pb):
    """Knockdown rows with at least MIN_CELLS cells (rows of unknown size are kept)."""
    n = np.asarray(pb['n_cells'], float)
    return np.flatnonzero((np.arange(len(n)) > 0) & ~(n < MIN_CELLS))

def lfc_stats(pb, chunk=1024):
    """Per gene: SD of log2FC across knockdowns. Per knockdown: #genes |log2FC|>1 among genes > 5 CPM in control.
    Only knockdowns with >= MIN_CELLS cells; `rows` says which pseudobulk rows the per-knockdown arrays refer to."""
    m = pb['mean_cpm']; ctrl = np.asarray(m[0], float); gated = np.nan_to_num(ctrl) > 5
    s1 = np.zeros(len(GENES)); s2 = np.zeros(len(GENES)); cnt = np.zeros(len(GENES))
    n_de, l2 = [], []; rows = kd_rows(pb)
    for b in range(0, len(rows), chunk):
        x = np.log2((np.asarray(m[rows[b:b + chunk]], float) + PC) / (ctrl + PC))
        ok = np.isfinite(x); xz = np.where(ok, x, 0)
        s1 += xz.sum(0); s2 += (xz ** 2).sum(0); cnt += ok.sum(0)
        n_de.append((np.abs(np.where(gated, xz, 0)) > 1).sum(1)); l2.append(np.sqrt((np.where(gated, xz, 0) ** 2).sum(1)))
    sd = np.sqrt(np.maximum(s2 / np.maximum(cnt, 1) - (s1 / np.maximum(cnt, 1)) ** 2, 0)); sd[cnt < 3] = np.nan
    return dict(sd=sd, rows=rows, n_de=np.concatenate(n_de) if n_de else np.array([]), l2=np.concatenate(l2) if l2 else np.array([]), gated=gated)

def expected_se(pb):
    ctrl = np.asarray(pb['mean_cpm'][0], float)
    if pb['kind'] == 'effects':
        return np.asarray(pb['mean_se'], float)
    n_kd = np.nanmedian(np.asarray(pb['n_cells'], float)[kd_rows(pb)]) if pb['n_cells'].shape[0] > 1 else np.nan
    var = np.asarray(pb['ctrl_var_cpm'], float)
    return np.sqrt(var / n_kd) / (ctrl + PC) / np.log(2)

STATS = {}
def plot_context(key):
    pb = PBS[key]; ctrl = np.asarray(pb['mean_cpm'][0], float)
    fig, ax = plt.subplots(1, 4, figsize=(19, 3.6)); fig.suptitle(f'{key}  ({pb["kind"]})', x=0.01, ha='left', fontweight='bold')
    n = np.asarray(pb['n_cells'], float)[1:]
    if len(n) and np.isfinite(n).any():
        ax[0].hist(n[np.isfinite(n)], bins=np.logspace(0, np.log10(np.nanmax(n) + 1), 40)); ax[0].set_xscale('log')
        ax[0].axvline(MIN_CELLS, color='r', ls=':', lw=1); ax[0].set_title(f'cells per knockdown (median {np.nanmedian(n):.0f}; {np.mean(n < MIN_CELLS):.1%} < {MIN_CELLS})')
    else:
        ax[0].text(.5, .5, 'controls only', ha='center', transform=ax[0].transAxes); ax[0].set_title('cells per knockdown')
    if 'cell_library' in pb:
        lib = np.asarray(pb['cell_library']); ax[1].hist(lib, bins=60); ax[1].set_title(f'UMIs per cell (median {np.median(lib):,.0f})')
    elif 'umi_per_cell' in pb:
        u = np.asarray(pb['umi_per_cell'])[1:]; ax[1].hist(u, bins=60); ax[1].set_title(f'mean UMIs/cell per knockdown (median {np.median(u):,.0f})')
    else:
        ax[1].text(.5, .5, 'no cell-level data released', ha='center', transform=ax[1].transAxes); ax[1].set_title('depth')
    if 'ctrl_var_cpm' in pb:
        var = np.asarray(pb['ctrl_var_cpm'], float); ok = (ctrl > 0) & (var > 0)
        ax[2].loglog(ctrl[ok], var[ok], '.', ms=1.5, alpha=.3)
        depth = np.median(np.asarray(pb['cell_library'])) if 'cell_library' in pb else float(np.asarray(pb['umi_per_cell'])[0])
        xs = np.logspace(np.log10(ctrl[ok].min()), np.log10(ctrl[ok].max()), 50)
        ax[2].loglog(xs, xs * 1e6 / depth, 'k--', lw=1, label='Poisson'); ax[2].legend(loc='upper left')
        ax[2].set_xlabel('control mean CPM'); ax[2].set_ylabel('variance of per-cell CPM'); ax[2].set_title('control mean-variance')
    else:
        ok = ctrl > 0; ax[2].hist(np.log10(ctrl[ok]), bins=60); ax[2].set_xlabel('log10 control CPM'); ax[2].set_title('control expression (no per-cell variance)')
    if pb['mean_cpm'].shape[0] > 3:
        st = lfc_stats(pb); se = expected_se(pb); STATS[key] = dict(st, se=se)
        bins = np.arange(-1, 4.01, .25); x = np.log10(ctrl + 1e-3); mid = (bins[1:] + bins[:-1]) / 2
        sd_b = [np.nanmedian(st['sd'][(x >= lo) & (x < hi)]) for lo, hi in zip(bins[:-1], bins[1:])]
        se_b = [np.nanmedian(se[(x >= lo) & (x < hi)]) for lo, hi in zip(bins[:-1], bins[1:])]
        ax[3].plot(mid, sd_b, label='spread across knockdowns'); ax[3].plot(mid, se_b, label='expected sampling SE')
        ax[3].axvline(np.log10(5), color='grey', ls=':', lw=1); ax[3].text(np.log10(5), ax[3].get_ylim()[1]*.9, ' 5 CPM gate', color='grey', fontsize=8)
        ax[3].set_xlabel('log10 control CPM'); ax[3].set_ylabel('log2 FC'); ax[3].set_title('signal vs noise'); ax[3].legend()
    else:
        ax[3].text(.5, .5, 'controls only:\nno knockdowns to compare', ha='center', va='center', transform=ax[3].transAxes); ax[3].set_title('signal vs noise')
    plt.tight_layout(); plt.show()

for key in ALL:
    plot_context(key)

## 3. Raw single-cell files
Every perturbation source we have (or are downloading) as raw single cells, before any cleaning. Each file is opened without loading it into memory. Shown per file: cells × genes and how the counts are stored, the first 20 cells' metadata (`obs`), the first 20 genes (`var`), and raw counts for those 20 cells over the 20 genes with the most counts in them (the first genes are often all zero).

Not here: CD4 T cells (effect tables only) and the full X-Atlas/Orion release (~520 GB; we hold HCT116 / HEK293T cells for the A–C targets only). A file not yet downloaded is listed as missing.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from IPython.display import display
V = Path.home() / 'vcc'
PB = V / 'lib/downloads/perturbase'
RAW = {
    'K562':     (V / 'lib/replogle/K562_gwps_raw_singlecell_01.h5ad', 'Replogle 2022, genome-wide CRISPRi'),
    'RPE1':     (V / 'lib/downloads/rpe1/rpe1_raw_singlecell_01.h5ad', 'Replogle 2022, essential genes'),
    'HepG2':    (V / 'lib/downloads/nadig_hepg2/GSE264667_hepg2_raw_singlecell_01.h5ad', 'Nadig 2025'),
    'Jurkat':   (V / 'lib/downloads/nadig_jurkat/GSE264667_jurkat_raw_singlecell_01.h5ad', 'Nadig 2025'),
    'H1':       (V / 'lib/h1/H1_full_2025.h5ad', 'Arc 2025 (our validation line)'),
    'HCT116':   (V / 'lib/orion/HCT116_cells_targets.h5ad', 'X-Atlas/Orion 2025, A-C targets + controls only'),
    'HEK293T':  (V / 'lib/orion/HEK293T_cells_targets.h5ad', 'X-Atlas/Orion 2025, A-C targets + controls only'),
    'hESC':     (PB / 'PRJNA524331/raw.h5ad', 'Genga 2019, during endoderm differentiation'),
    'HFF':      (PB / 'CRISPRi_perturb_host/raw.h5ad', 'Hein 2021, foreskin fibroblasts, viral-infection screen'),
    'MCF10A_Dox':  (PB / 'Dox/raw.h5ad', 'Hill 2018, doxorubicin; singles and pairs'),
    'MCF10A_Mock': (PB / 'Mock/raw.h5ad', 'Hill 2018, untreated; singles and pairs'),
    'LUHMES':   (PB / 'PRJNA595864/raw.h5ad', 'Lalli 2020'),
}

def show_raw(name, n=20):
    path, note = RAW[name]
    if not path.exists():
        print(f'{name}: not on disk yet ({path})'); return
    a = ad.read_h5ad(path, backed='r')
    X = a.X
    kind = 'sparse (CSR)' if hasattr(X, 'format') or 'sparse' in type(X).__name__.lower() else 'dense'
    print(f'{name} - {note}\n{path}\n{a.n_obs:,} cells x {a.n_vars:,} genes; counts stored {kind}, {X.dtype}; {path.stat().st_size / 1e9:.1f} GB')
    print(f'obs columns: {list(a.obs.columns)}\nvar columns: {list(a.var.columns)}')
    display(a.obs.head(n)); display(a.var.head(n))
    block = X[:n]
    block = block.toarray() if sp.issparse(block) or hasattr(block, 'toarray') else np.asarray(block)
    names = a.var['gene_name'].astype(str).to_numpy() if 'gene_name' in a.var else a.var_names.astype(str).to_numpy()
    top = np.argsort(-block.sum(0), kind='stable')[:n]
    print(f'raw counts: first {n} cells x the {n} genes with the most counts in them')
    display(pd.DataFrame(block[:, top], index=a.obs_names[:n], columns=names[top]))
    a.file.close()

pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
pd.DataFrame([(k, v[1], v[0].exists(), round(v[0].stat().st_size / 1e9, 2) if v[0].exists() else None) for k, v in RAW.items()],
             columns=['dataset', 'source', 'on disk', 'GB'])

In [ ]:
show_raw('K562')

In [ ]:
show_raw('RPE1')

In [ ]:
show_raw('HepG2')

In [ ]:
show_raw('Jurkat')

In [ ]:
show_raw('H1')

In [ ]:
show_raw('HCT116')

In [ ]:
show_raw('HEK293T')

In [ ]:
show_raw('hESC')

In [ ]:
show_raw('HFF')

In [ ]:
show_raw('MCF10A_Dox')

In [ ]:
show_raw('MCF10A_Mock')

In [ ]:
show_raw('LUHMES')

### 3b. How usable are the PerturBase label sets?
Per file: cells by label kind (control, single knockdown, pair, 3+ guides, no guide), split at 1,000 UMIs to separate real cells from near-empty droplets, and how many single knockdowns reach **30 cells** (our QC). HFF, hESC, MCF10A and LUHMES are already cell-filtered. (The Tian 2019 / 2021 iPSC and neuron screens were reviewed and removed on 2026-10-18: most knockdowns lowered their target by less than 50%, and Tian 2019 was unfiltered droplet output.) Note that three files store counts column by column (CSC); the UMI sums below handle both layouts.

In [ ]:
import h5py
from anndata.io import read_elem

def cell_umis(path):
    """Total UMIs per cell from the file's own layout (CSR = by cell, CSC = by gene), read in chunks."""
    with h5py.File(path, 'r') as f:
        X = f['X']; n = int(X.attrs['shape'][0]); ip = X['indptr'][:]; out = np.zeros(n)
        if X.attrs.get('encoding-type') == 'csc_matrix':
            for a in range(0, int(ip[-1]), 5_000_000):
                b = min(a + 5_000_000, int(ip[-1]))
                out += np.bincount(X['indices'][a:b], weights=X['data'][a:b].astype(float), minlength=n)
        else:
            for s in range(0, n, 20_000):
                e = min(s + 20_000, n); a, b = int(ip[s]), int(ip[e])
                out[s:e] = np.bincount(np.repeat(np.arange(e - s), np.diff(ip[s:e + 1])), weights=X['data'][a:b].astype(float), minlength=e - s)
    return out

def label_kinds(labels):
    n = labels.map(lambda s: len([p for p in s.split(',') if p]))
    return pd.Series(np.select([labels.isin(['None', 'nan', '']), labels == 'CTRL', n == 1, n == 2],
                               ['no guide', 'control', 'single', 'pair'], '3+ guides'), index=labels.index)

def label_summary(name, min_cells=30, min_umi=1000):
    path = RAW[name][0]
    if not path.exists(): print(f'{name}: not on disk'); return None
    with h5py.File(path, 'r') as f: labels = read_elem(f['obs'])['gene'].astype(str).reset_index(drop=True)
    umi = cell_umis(path); kind = label_kinds(labels); real = umi >= min_umi
    print(f'== {name}: {len(labels):,} barcodes, {real.sum():,} with >= {min_umi:,} UMIs; median UMI {np.median(umi):,.0f}')
    display(pd.crosstab(kind, np.where(real, f'>= {min_umi} UMI', f'< {min_umi} UMI'), margins=True))
    singles = labels[(kind == 'single') & real].value_counts()
    print(f'single knockdowns in real cells: {len(singles)} genes, {int((singles >= min_cells).sum())} with >= {min_cells} cells; '
          f'controls: {int(((kind == "control") & real).sum()):,} cells')
    display(singles.rename('cells').to_frame().T)
    return dict(dataset=name, barcodes=len(labels), real_cells=int(real.sum()), controls=int(((kind == 'control') & real).sum()),
                single_cells=int(((kind == 'single') & real).sum()), knockdowns=len(singles), knockdowns_ge30=int((singles >= min_cells).sum()))

summary = [label_summary(n) for n in ['HFF', 'hESC', 'MCF10A_Dox', 'MCF10A_Mock', 'LUHMES']]
pd.DataFrame([s for s in summary if s]).set_index('dataset')